In [30]:
import os
import numpy as np
import csv

order = ['sad', 'dis', 'fear', 'neu', 'joy', 'ten', 'ins']
nums = ['4', '5', '8']

trial_label2 = []
path = r'/mnt/dataset4/sitian/post_training/danmu/EmoEEG-MC/results/smooth_scores'
mode1_rows = np.full((21, 200, 7), np.nan)
mode2_rows = np.full((21, 200, 7), np.nan)
mode3_rows = np.full((21, 200, 7), np.nan)

for i in range(len(order)):
    for j in range(3):
        mode1_name = os.path.join(path, f'{order[i]}{nums[j]}_smooth_mode1.csv')
        mode2_name = os.path.join(path, f'{order[i]}{nums[j]}_smooth_mode2.csv')
        mode3_name = os.path.join(path, f'{order[i]}{nums[j]}_smooth_mode3.csv')
        # 判断文件是否存在
        if os.path.exists(mode1_name):
            with open(mode1_name, 'r') as f:
                reader = csv.reader(f)
                mode1_rows_tr = [row for row in reader]
                for k in range(len(mode1_rows_tr[1:])):
                    for l in range(7):
                        mode1_rows[i*3+j][k][l] = float(mode1_rows_tr[k+1][l])
        if os.path.exists(mode2_name):
            with open(mode2_name, 'r') as f:
                reader = csv.reader(f)
                mode2_rows_tr = [row for row in reader]
                for k in range(len(mode2_rows_tr[1:])):
                    for l in range(7):
                        mode2_rows[i*3+j][k][l] = float(mode2_rows_tr[k+1][l])
        if os.path.exists(mode3_name):
            with open(mode3_name, 'r') as f:
                reader = csv.reader(f)
                mode3_rows_tr = [row for row in reader]
                for k in range(len(mode3_rows_tr[1:])):
                    for l in range(7):
                        mode3_rows[i*3+j][k][l] = float(mode3_rows_tr[k+1][l])

In [31]:
# (21, 100, 21)，将三个mode_rows在第三维合并
trial_label2 = np.concatenate((mode1_rows, mode2_rows, mode3_rows), axis=2)
print(trial_label2.shape)  # (21, 100, 21)

# 将trial_label1扩展为(21, 100, 1)
trial_label1 = [0]*3 + [1]*3 + [2]*3 + [3]*3 + [4]*3 + [5]*3 + [6]*3
trial_label1 = np.array(trial_label1).reshape(21, 1, 1)
trial_label1 = np.tile(trial_label1, (1, 200, 1))
print(trial_label1.shape)  # (21, 100, 1)

(21, 200, 21)
(21, 200, 1)


In [32]:
trial_labels = np.concatenate((trial_label1, trial_label2), axis=2)
print(trial_labels.shape)  # (21, 100, 22)

(21, 200, 22)


In [ ]:

import os
import re
import numpy as np
import mne
from scipy.io import savemat
from collections import defaultdict
import h5py
from pathlib import Path
import pickle

print(np.__version__)
# 1.24.4
print(mne.__version__)
# 0.23.0

def stretch_axis(arr, T_prime):
    is_1d = arr.ndim == 1
    if is_1d:
        arr = arr[np.newaxis, :] # 变为 [1, k]
    
    T = arr.shape[-1]
    x_old = np.arange(T)
    x_new = np.linspace(0, T - 1, T_prime)
    
    # 对每一行执行插值
    res = np.apply_along_axis(lambda y: np.interp(x_new, x_old, y), axis=-1, arr=arr)
    
    return res.flatten() if is_1d else res


# ======================= 配置 =======================
dataset_name = 'EmoEEG-MC_cla'
target_sfreq = 200
sample_T = 2.0 # EEG sample window timelength in s
sample_stride = 2.0 # # EEG sample window stride in s

order = ['sad', 'dis', 'fear', 'neu', 'joy', 'ten', 'ins']
nums = ['4', '5', '8']

input_dir = r"/mnt/dataset0/xuxin/new_pre/reorder"
output_dir = f"/mnt/dataset2/Processed_datasets/EEG_Bench/EmoEEG-MC_reg"
# output_dir = f"/mnt/dataset4/daily_eeg_emotion/Data/EmoEEG-MC/EmoEEG-MC_hdf5_test"
sample_samples = int(sample_T * target_sfreq)
stride_samples = int(sample_stride * target_sfreq)

n_sub = 53

subject_files = defaultdict(list)

for i in range(46, n_sub):
    print(f"Processing subject {i+2}/{n_sub}")
    file_sub = []
    for j in range(len(order)):
        for k in range(3):
            # sub_dir_ima = os.path.join(input_dir, f'sub_{i+2}', f'sub-{i+2:02d}_ses-ima_task-emotion_{order[j]}{nums[k]}.npy')
            sub_dir_vid = os.path.join(input_dir, f'sub_{i+2}', f'sub-{i+2:02d}_ses-vid_task-emotion_{order[j]}{nums[k]}.npy')
            # if (not os.path.exists(sub_dir_ima)) or (not os.path.exists(sub_dir_vid)):
            #     print(f"File not found: {sub_dir_ima} or {sub_dir_vid}")
            #     continue
            # if (os.path.getsize(sub_dir_ima) == 0) or (os.path.getsize(sub_dir_vid) == 0):
            #     print(f"File is empty: {sub_dir_ima} or {sub_dir_vid}")
            #     continue
            if(not os.path.exists(sub_dir_vid)):
                print(f"File not found: {sub_dir_vid}")
                continue
            if(os.path.getsize(sub_dir_vid) == 0):
                print(f"File is empty: {sub_dir_vid}")
                continue
            # data_ima = np.load(sub_dir_ima)  # (62, T)
            data_vid = np.load(sub_dir_vid)  # (62, T)
            # file_sub.append(data_ima)
            file_sub.append(data_vid)
    subject_files[i] = file_sub
print(f"Total subjects: {len(subject_files)}")

# ======================= 21 个 trial 标签 =======================
# 将(21, 100, 22)的trial_labels扩展为(n_sub, 21, 100, 22)
all_labels = np.tile(trial_labels[np.newaxis, :, :, :], (n_sub, 1, 1, 1))
print(all_labels.shape)  # (n_sub, 21, 22)

# ======================= 主处理 =======================
ch_names = [
    'Fp1', 'Fpz', 'Fp2', 'AF7', 'AF3','AF4','AF8', 'F7', 'F5','F3','F1','Fz', 'F2', 'F4', 'F6', 'F8',
    'FT7', 'FC5', 'FC3', 'FC1','FCz','FC2','FC4', 'FC6', 'FT8', 'T7','C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'T8',
    'TP7', 'CP5', 'CP3', 'CP1','CPz','CP2', 'CP4','CP6', 'TP8', 'P7','P5', 'P3', 'P1', 'Pz','P2', 'P4', 'P6', 'P8',
    'PO7', 'PO3','POz', 'PO4','PO8', 'O1','Oz','O2', 'F9', 'F10', 'TP9', 'TP10'
]

output_path_base = Path(output_dir)
output_path_base.mkdir(parents=True, exist_ok=True)
subs = None
subs = []

for sub in range(46, n_sub):
    # if(subs is not None and sub not in subs):
    #     continue
    sub_str = str(sub+2)
    
    sub_h5_path = output_path_base / f"sub_{sub+2}.h5"
    with h5py.File(sub_h5_path, 'w') as f:
            print(sub)
            # if(sub != '7'):
            #     continue
            print(f"\n处理 sub_{sub}")
            trial_segments = subject_files[sub]  # (21, 64,T)
            print(len(trial_segments))    # trial_segments: (42, 64, T)
            print(ch_names)
            
            for i_trial, data_trial in enumerate(trial_segments):
                if(i_trial < 21):
                    session = 1
                else:
                    session = 2
                print(f'sub {sub} trial {i_trial}: {data_trial.shape}')
                trial_grp = f.create_group(f"trial{i_trial}")
                n_samples = data_trial.shape[-1]
                sub_trial_label = all_labels[sub][i_trial]
                n_slices = (n_samples - sample_samples + stride_samples) // stride_samples
                sub_trial_label = np.array(sub_trial_label)
                print(sub_trial_label)  # (22,)
                # sub_trial_label = np.tile(sub_trial_label, n_slices)
                for i_slice, start in enumerate(range(0, n_samples - sample_samples + 1, stride_samples)):
                    end = start + sample_samples
                    slice_data = data_trial[:, start:end]
                    slice_grp = trial_grp.create_group(f"sample{i_slice}")
                    dset = slice_grp.create_dataset('eeg', data=slice_data)
                    dset.attrs['rsFreq'] = target_sfreq
                    dset.attrs['label'] = np.array([sub_trial_label[i_slice]])
                    dset.attrs['subject_id'] = sub
                    dset.attrs['trial_id'] = i_trial
                    dset.attrs['session_id'] = session
                    dset.attrs['segment_id'] = i_slice
                    dset.attrs['time_length'] = sample_T
                    dset.attrs['dataset_name'] = 'EmoEEG-MC_reg'
                    dset.attrs['chn_name'] = ch_names
                    dset.attrs['chn_pos'] = 'None'
                    dset.attrs['chn_ori'] = 'None'
                    dset.attrs['chn_type'] = 'EEG'
                    dset.attrs['montage'] = '10_20'

2.2.3
1.11.0
Processing subject 2/53
Processing subject 3/53
Processing subject 4/53
Processing subject 5/53
Processing subject 6/53
Processing subject 7/53
Processing subject 8/53
Processing subject 9/53
Processing subject 10/53
Processing subject 11/53
Processing subject 12/53
Processing subject 13/53
Processing subject 14/53
Processing subject 15/53
Processing subject 16/53
Processing subject 17/53
Processing subject 18/53
Processing subject 19/53
Processing subject 20/53
Processing subject 21/53
Processing subject 22/53
Processing subject 23/53
Processing subject 24/53
Processing subject 25/53
Processing subject 26/53
Processing subject 27/53
Processing subject 28/53
Processing subject 29/53
Processing subject 30/53
Processing subject 31/53
Processing subject 32/53
Processing subject 33/53
Processing subject 34/53
Processing subject 35/53
Processing subject 36/53
Processing subject 37/53
Processing subject 38/53
Processing subject 39/53
Processing subject 40/53
Processing subject 4

IndexError: index 200 is out of bounds for axis 0 with size 200

In [ ]:
import os
import re
import numpy as np
import mne
from scipy.io import savemat
from collections import defaultdict
import h5py
from pathlib import Path
import pickle

print(np.__version__)
# 1.24.4
print(mne.__version__)
# 0.23.0

def stretch_axis(arr, T_prime):
    is_1d = arr.ndim == 1
    if is_1d:
        arr = arr[np.newaxis, :] # 变为 [1, k]
    
    T = arr.shape[-1]
    x_old = np.arange(T)
    x_new = np.linspace(0, T - 1, T_prime)
    
    # 对每一行执行插值
    res = np.apply_along_axis(lambda y: np.interp(x_new, x_old, y), axis=-1, arr=arr)
    
    return res.flatten() if is_1d else res


# ======================= 配置 =======================
dataset_name = 'EmoEEG-MC_cla'
target_sfreq = 200
sample_T = 2.0 # EEG sample window timelength in s
sample_stride = 2.0 # # EEG sample window stride in s

order = ['sad', 'dis', 'fear', 'neu', 'joy', 'ten', 'ins']
nums = ['4', '5', '8']

input_dir = r"/mnt/dataset0/xuxin/new_pre/reorder"
output_dir = f"/mnt/dataset2/Processed_datasets/EEG_Bench/EmoEEG-MC_cla"
# output_dir = f"/mnt/dataset4/daily_eeg_emotion/Data/EmoEEG-MC/EmoEEG-MC_hdf5_test"
sample_samples = int(sample_T * target_sfreq)
stride_samples = int(sample_stride * target_sfreq)

n_sub = 53

subject_files = defaultdict(list)

for i in range(n_sub):
    print(f"Processing subject {i+2}/{n_sub}")
    file_sub = []
    for j in range(len(order)):
        for k in range(3):
            sub_dir_ima = os.path.join(input_dir, f'sub_{i+2}', f'sub-{i+2:02d}_ses-ima_task-emotion_{order[j]}{nums[k]}.npy')
            sub_dir_vid = os.path.join(input_dir, f'sub_{i+2}', f'sub-{i+2:02d}_ses-vid_task-emotion_{order[j]}{nums[k]}.npy')
            if (not os.path.exists(sub_dir_ima)) or (not os.path.exists(sub_dir_vid)):
                print(f"File not found: {sub_dir_ima} or {sub_dir_vid}")
                continue
            if (os.path.getsize(sub_dir_ima) == 0) or (os.path.getsize(sub_dir_vid) == 0):
                print(f"File is empty: {sub_dir_ima} or {sub_dir_vid}")
                continue
            data_ima = np.load(sub_dir_ima)  # (62, T)
            data_vid = np.load(sub_dir_vid)  # (62, T)
            file_sub.append(data_ima)
            file_sub.append(data_vid)
    subject_files[i] = file_sub
print(f"Total subjects: {len(subject_files)}")

# ======================= 21 个 trial 标签 =======================
label_dim = 7
trial_label_ima = [0]*3 + [1]*3 + [2]*3 + [3]*3 + [4]*3 + [5]*3 + [6]*3
trial_labels_vid = [0]*3 + [1]*3 + [2]*3 + [3]*3 + [4]*3 + [5]*3 + [6]*3
trial_label = trial_label_ima + trial_labels_vid  # length: 42
trial_labels = np.array(trial_label)

all_labels = np.tile(trial_labels, (n_sub, 1))
print(all_labels.shape)  # (n_sub, 42, 1)

# ======================= 主处理 =======================
ch_names = [
    'Fp1', 'Fpz', 'Fp2', 'AF7', 'AF3','AF4','AF8', 'F7', 'F5','F3','F1','Fz', 'F2', 'F4', 'F6', 'F8',
    'FT7', 'FC5', 'FC3', 'FC1','FCz','FC2','FC4', 'FC6', 'FT8', 'T7','C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'T8',
    'TP7', 'CP5', 'CP3', 'CP1','CPz','CP2', 'CP4','CP6', 'TP8', 'P7','P5', 'P3', 'P1', 'Pz','P2', 'P4', 'P6', 'P8',
    'PO7', 'PO3','POz', 'PO4','PO8', 'O1','Oz','O2', 'F9', 'F10', 'TP9', 'TP10'
]

output_path_base = Path(output_dir)
output_path_base.mkdir(parents=True, exist_ok=True)
subs = None
subs = []

for sub in range(n_sub):
    # if(subs is not None and sub not in subs):
    #     continue
    sub_str = str(sub+2)
    
    sub_h5_path = output_path_base / f"sub_{sub+2}.h5"
    with h5py.File(sub_h5_path, 'w') as f:
            print(sub)
            # if(sub != '7'):
            #     continue
            print(f"\n处理 sub_{sub}")
            trial_segments = subject_files[sub]  # (42, 64,T)
            print(len(trial_segments))    # trial_segments: (42, 64, T)
            print(ch_names)
            
            for i_trial, data_trial in enumerate(trial_segments):
                if(i_trial < 21):
                    session = 1
                else:
                    session = 2
                print(f'sub {sub} trial {i_trial}: {data_trial.shape}')
                trial_grp = f.create_group(f"trial{i_trial}")
                n_samples = data_trial.shape[-1]
                sub_trial_label = all_labels[sub][i_trial]
                n_slices = (n_samples - sample_samples + stride_samples) // stride_samples
                sub_trial_label = np.array(sub_trial_label)
                sub_trial_label = np.tile(sub_trial_label, n_slices)
                for i_slice, start in enumerate(range(0, n_samples - sample_samples + 1, stride_samples)):
                    end = start + sample_samples
                    slice_data = data_trial[:, start:end]
                    slice_grp = trial_grp.create_group(f"sample{i_slice}")
                    dset = slice_grp.create_dataset('eeg', data=slice_data)
                    dset.attrs['rsFreq'] = target_sfreq
                    dset.attrs['label'] = np.array([sub_trial_label[i_slice]])
                    dset.attrs['subject_id'] = sub
                    dset.attrs['trial_id'] = i_trial
                    dset.attrs['session_id'] = session
                    dset.attrs['segment_id'] = i_slice
                    dset.attrs['time_length'] = sample_T
                    dset.attrs['dataset_name'] = 'EmoEEG-MC_cla'
                    dset.attrs['chn_name'] = ch_names
                    dset.attrs['chn_pos'] = 'None'
                    dset.attrs['chn_ori'] = 'None'
                    dset.attrs['chn_type'] = 'EEG'
                    dset.attrs['montage'] = '10_20'